In [1]:
!pip install textattack[tensorflow]
!pip install transformers==4.41.0

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 55.5/55.5 kB 5.0 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.6/3.6 MB 48.0 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 981.5/981.5 kB 43.6 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
  Preparing metadata (setup.py) ... done
  Preparing metadata (setup.py) ... done
  Preparing metadata (setup.py) ... done
  Preparing metadata (setup.py) ... done
  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 61.1/61.1 kB 2.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 442.0/442.0 kB 30.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 363.4/363.4 MB 4.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 13.8/13.8 MB 95.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 24.6/24.6 MB 78.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

In [6]:
!wget --no-check-certificate http://nlp.stanford.edu/data/glove.6B.zip
!unzip glove.6B.zip

--2025-06-14 18:21:15--  http://nlp.stanford.edu/data/glove.6B.zip
Resolving nlp.stanford.edu (nlp.stanford.edu)... 171.64.67.140
Connecting to nlp.stanford.edu (nlp.stanford.edu)|171.64.67.140|:80... connected.
HTTP request sent, awaiting response... 302 Found
Location: https://nlp.stanford.edu/data/glove.6B.zip [following]
--2025-06-14 18:21:15--  https://nlp.stanford.edu/data/glove.6B.zip
Connecting to nlp.stanford.edu (nlp.stanford.edu)|171.64.67.140|:443... connected.
HTTP request sent, awaiting response... 301 Moved Permanently
Location: https://downloads.cs.stanford.edu/nlp/data/glove.6B.zip [following]
--2025-06-14 18:21:15--  https://downloads.cs.stanford.edu/nlp/data/glove.6B.zip
Resolving downloads.cs.stanford.edu (downloads.cs.stanford.edu)... 171.64.64.22
Connecting to downloads.cs.stanford.edu (downloads.cs.stanford.edu)|171.64.64.22|:443... connected.
  Issued certificate has expired.
HTTP request sent, awaiting response... 200 OK
Length: 862182613 (822M) [application/zi

In [7]:
import numpy as np
import matplotlib.pyplot as plt
from sklearn.decomposition import PCA

# Load GloVe embeddings
glove_path = "/content/glove.6B.100d.txt"
embedding_index = {}

with open(glove_path, "r", encoding="utf-8") as f:
    for line in f:
        values = line.split()
        word = values[0]
        vector = np.asarray(values[1:], dtype="float32")
        embedding_index[word] = vector



In [8]:
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras.layers import Embedding, LSTM, Dense, Dropout, Bidirectional, GlobalAveragePooling1D
from tensorflow.keras.preprocessing.sequence import pad_sequences
from tensorflow.keras.regularizers import l2
import tensorflow_datasets as tfds
import numpy as np
from tensorflow.keras.models import load_model
import pickle



# Load the saved tokenizer from the pickle file
with open("tokenizer.pickle", "rb") as handle:
   tokenizer = pickle.load(handle)

# Load Yelp Polarity dataset
dataset, info = tfds.load('yelp_polarity_reviews', with_info=True, as_supervised=True)
train_data, test_data = dataset['train'], dataset['test']

# Prepare the data
vocab_size = 50000  # Limit vocabulary to top 50,000 words
max_length = 200    # Maximum review length (truncation/padding)

# Tokenizer to convert text to sequences
#tokenizer = keras.preprocessing.text.Tokenizer(num_words=vocab_size)
#tokenizer.fit_on_texts([text.numpy().decode('utf-8') for text, _ in train_data])

# Convert text data to sequences and pad them
def preprocess_dataset(dataset):
    texts, labels = [], []
    for text, label in dataset:
        texts.append(text.numpy().decode('utf-8'))
        labels.append(label.numpy())
    sequences = tokenizer.texts_to_sequences(texts)
    padded_sequences = pad_sequences(sequences, maxlen=max_length, padding='post', truncating='post')
    return padded_sequences, np.array(labels)

# Preprocess training and testing data
tr_x, tr_y = preprocess_dataset(train_data)
te_x, te_y = preprocess_dataset(test_data)



Dl Completed...: 0 url [00:00, ? url/s]

Dl Size...: 0 MiB [00:00, ? MiB/s]

Extraction completed...: 0 file [00:00, ? file/s]

Generating splits...:   0%|          | 0/2 [00:00<?, ? splits/s]

Generating train examples...: 0 examples [00:00, ? examples/s]

Shuffling /root/tensorflow_datasets/yelp_polarity_reviews/incomplete.NJDFVF_0.2.0/yelp_polarity_reviews-train.…

Generating test examples...: 0 examples [00:00, ? examples/s]

Shuffling /root/tensorflow_datasets/yelp_polarity_reviews/incomplete.NJDFVF_0.2.0/yelp_polarity_reviews-test.t…

Dataset yelp_polarity_reviews downloaded and prepared to /root/tensorflow_datasets/yelp_polarity_reviews/0.2.0. Subsequent calls will reuse this data.


In [9]:
# Load the model
model = load_model('NLPModel.h5')

In [10]:
vocab_size = 50000
# Load GloVe 6B embeddings
glove_path = "/content/glove.6B.100d.txt"  # Update with the correct path
embedding_dim = 100  # Choose dimension (50d, 100d, 200d, or 300d)

# Create a dictionary mapping words to vectors
embeddings_index = {}
with open(glove_path, "r", encoding="utf-8") as f:
    for line in f:
        values = line.split()
        word = values[0]
        vector = np.asarray(values[1:], dtype="float32")
        embeddings_index[word] = vector

# Create an embedding matrix for the tokenizer’s vocabulary
embedding_matrix = np.zeros((vocab_size, embedding_dim))
for word, i in tokenizer.word_index.items():
    if i < vocab_size:
        embedding_vector = embeddings_index.get(word)
        if embedding_vector is not None:
            embedding_matrix[i] = embedding_vector



In [11]:
def predict_sentiment(model, text, tokenizer, max_length=200):
    # Convert text to sequence using the tokenizer
    sequence = tokenizer.texts_to_sequences([text])

    # Pad the sequence to the fixed length
    padded_sequence = pad_sequences(sequence, maxlen=max_length, padding='post', truncating='post')

    # Get prediction probability
    prediction = model.predict(padded_sequence)[0][0]

    # Classify sentiment
    sentiment = "Positive" if prediction > 0.5 else "Negative"

    # Confidence score
    confidence = prediction if sentiment == "Positive" else 1 - prediction

    return sentiment, confidence


In [12]:
import numpy as np
import tensorflow as tf
import textattack
from textattack import Attack, Attacker, AttackArgs
from textattack.constraints.pre_transformation import (
    RepeatModification,
    StopwordModification,
    InputColumnModification
)
from textattack.constraints.grammaticality import PartOfSpeech
from textattack.constraints.semantics import WordEmbeddingDistance
from textattack.constraints.semantics.sentence_encoders import UniversalSentenceEncoder

from textattack.transformations import WordSwapEmbedding
from textattack.search_methods import GreedyWordSwapWIR
from textattack.goal_functions import UntargetedClassification
from textattack.models.wrappers import ModelWrapper
from textattack.shared import WordEmbedding
from tensorflow.keras.preprocessing.sequence import pad_sequences

textattack: Updating TextAttack package dependencies.
textattack: Downloading NLTK required packages.
[nltk_data] Downloading package averaged_perceptron_tagger to
[nltk_data]     /root/nltk_data...
[nltk_data]   Unzipping taggers/averaged_perceptron_tagger.zip.
[nltk_data] Downloading package stopwords to /root/nltk_data...
[nltk_data]   Unzipping corpora/stopwords.zip.
[nltk_data] Downloading package omw to /root/nltk_data...
[nltk_data] Downloading package universal_tagset to /root/nltk_data...
[nltk_data]   Unzipping taggers/universal_tagset.zip.
[nltk_data] Downloading package wordnet to /root/nltk_data...
[nltk_data] Downloading package punkt to /root/nltk_data...
[nltk_data]   Unzipping tokenizers/punkt.zip.


In [13]:
from textattack.models.wrappers import ModelWrapper
import numpy as np
class CustomKerasModelWrapper(ModelWrapper):
    def __init__(self, model, tokenizer, max_length=200):
        self.model = model
        self.tokenizer = tokenizer
        self.max_length = max_length

    def __call__(self, text_input_list):
        sequences = self.tokenizer.texts_to_sequences(text_input_list)
        padded = pad_sequences(sequences, maxlen=self.max_length, padding='post', truncating='post')

        # Get raw probabilities (shape [batch_size, 1])
        preds = self.model.predict(padded)

        # Convert to logits for both classes (shape [batch_size, 2])
        # For binary classification: class 0 = 1 - prob, class 1 = prob
        return np.concatenate([1 - preds, preds], axis=1)


model_wrapper = CustomKerasModelWrapper(model, tokenizer, max_length=max_length)
# Constraints with corrected imports
stopwords = set(
            ["a", "about", "above", "across", "after", "afterwards", "again", "against", "ain", "all", "almost", "alone", "along", "already", "also", "although", "am", "among", "amongst", "an", "and", "another", "any", "anyhow", "anyone", "anything", "anyway", "anywhere", "are", "aren", "aren't", "around", "as", "at", "back", "been", "before", "beforehand", "behind", "being", "below", "beside", "besides", "between", "beyond", "both", "but", "by", "can", "cannot", "could", "couldn", "couldn't", "d", "didn", "didn't", "doesn", "doesn't", "don", "don't", "down", "due", "during", "either", "else", "elsewhere", "empty", "enough", "even", "ever", "everyone", "everything", "everywhere", "except", "first", "for", "former", "formerly", "from", "hadn", "hadn't", "hasn", "hasn't", "haven", "haven't", "he", "hence", "her", "here", "hereafter", "hereby", "herein", "hereupon", "hers", "herself", "him", "himself", "his", "how", "however", "hundred", "i", "if", "in", "indeed", "into", "is", "isn", "isn't", "it", "it's", "its", "itself", "just", "latter", "latterly", "least", "ll", "may", "me", "meanwhile", "mightn", "mightn't", "mine", "more", "moreover", "most", "mostly", "must", "mustn", "mustn't", "my", "myself", "namely", "needn", "needn't", "neither", "never", "nevertheless", "next", "no", "nobody", "none", "noone", "nor", "not", "nothing", "now", "nowhere", "o", "of", "off", "on", "once", "one", "only", "onto", "or", "other", "others", "otherwise", "our", "ours", "ourselves", "out", "over", "per", "please", "s", "same", "shan", "shan't", "she", "she's", "should've", "shouldn", "shouldn't", "somehow", "something", "sometime", "somewhere", "such", "t", "than", "that", "that'll", "the", "their", "theirs", "them", "themselves", "then", "thence", "there", "thereafter", "thereby", "therefore", "therein", "thereupon", "these", "they", "this", "those", "through", "throughout", "thru", "thus", "to", "too", "toward", "towards", "under", "unless", "until", "up", "upon", "used", "ve", "was", "wasn", "wasn't", "we", "were", "weren", "weren't", "what", "whatever", "when", "whence", "whenever", "where", "whereafter", "whereas", "whereby", "wherein", "whereupon", "wherever", "whether", "which", "while", "whither", "who", "whoever", "whole", "whom", "whose", "why", "with", "within", "without", "won", "won't", "would", "wouldn", "wouldn't", "y", "yet", "you", "you'd", "you'll", "you're", "you've", "your", "yours", "yourself", "yourselves"]
        )
transformation = WordSwapEmbedding(max_candidates=50)
use_constraint = UniversalSentenceEncoder(
            threshold=0.840845057,
            metric="angular",
            compare_against_original=True,
            window_size=15,
            skip_text_shorter_than_window=True,
        )
constraints = [

    RepeatModification(),
    StopwordModification(stopwords=stopwords),
    #input_column_modification,
    WordEmbeddingDistance(min_cos_sim=0.5),
    PartOfSpeech(allow_verb_noun_swap=False),
    use_constraint
]

attack = Attack(
    UntargetedClassification(model_wrapper),
    constraints,
    transformation,
    GreedyWordSwapWIR(wir_method="delete")
)

textattack: Downloading https://textattack.s3.amazonaws.com/word_embeddings/paragramcf.
100%|██████████| 481M/481M [00:08<00:00, 58.1MB/s]
textattack: Unzipping file /root/.cache/textattack/tmp8noq4k4f.zip to /root/.cache/textattack/word_embeddings/paragramcf.
textattack: Successfully saved word_embeddings/paragramcf to cache.
textattack: Unknown if model of class <class 'keras.src.models.sequential.Sequential'> compatible with goal function <class 'textattack.goal_functions.classification.untargeted_classification.UntargetedClassification'>.


In [14]:
import numpy as np

# Take first 100 preprocessed samples
sample_size = 500
te_x_sample = te_x[:sample_size]
te_y_sample = te_y[:sample_size]

# Reconstruct raw texts from tokenizer (approximate)
reverse_word_map = dict(map(reversed, tokenizer.word_index.items()))
test_texts = [
    " ".join([reverse_word_map.get(idx, "") for idx in seq if idx != 0])
    for seq in te_x_sample
]


In [15]:
# Original predictions
baseline_preds = model.predict(te_x_sample)
baseline_accuracy = np.mean(np.round(baseline_preds).flatten() == te_y_sample)
print(f"Baseline Accuracy: {baseline_accuracy:.2%}")


16/16 ━━━━━━━━━━━━━━━━━━━━ 3s 115ms/step
Baseline Accuracy: 95.40%


In [16]:
from textattack.datasets import Dataset
import nltk
from textattack.attack_results import (
    AttackResult,
    SuccessfulAttackResult,
    FailedAttackResult,
    SkippedAttackResult
)
nltk.download('averaged_perceptron_tagger_eng')
# Filter to samples the model originally got RIGHT
original_correct_mask = (np.round(baseline_preds).flatten() == te_y_sample)
originally_correct_indices = np.where(original_correct_mask)[0]
num_originally_correct = len(originally_correct_indices)

print(f"Samples originally correct: {num_originally_correct}/{sample_size}")

# Subset to only these correctly-classified samples
attack_texts = [test_texts[i] for i in originally_correct_indices]
attack_labels = te_y_sample[originally_correct_indices]
attack_labels_fixed = [int(label) for label in attack_labels]  # Convert to Python int

# Attack only these samples
from textattack import Attacker, AttackArgs

attack_args = AttackArgs(
    num_examples=num_originally_correct,
    log_to_csv="attack_log.csv",
    disable_stdout=True
)
attacker = Attacker(attack, Dataset(list(zip(attack_texts, attack_labels_fixed))), attack_args)  # Use fixed labels
attack_results = attacker.attack_dataset()


# Calculate metrics per paper
successful_attacks = sum(1 for res in attack_results if isinstance(res, SuccessfulAttackResult))
skipped_attacks = sum(1 for res in attack_results if isinstance(res, SkippedAttackResult))
failed_attacks = num_originally_correct - successful_attacks - skipped_attacks

# Paper-style metrics
original_accuracy = num_originally_correct / sample_size  # Baseline accuracy
attack_success_rate = successful_attacks / (num_originally_correct - skipped_attacks)  # ASR
accuracy_under_attack = (num_originally_correct - successful_attacks) / sample_size  # AuA

print(f"""
+---------------------------+-----------+
| Metric                    | Value     |
+---------------------------+-----------+
| Original Accuracy         | {original_accuracy:.2%}  |
| Accuracy Under Attack     | {accuracy_under_attack:.2%}  |
| Attack Success Rate (ASR) | {attack_success_rate:.2%}  |
| Successful Attacks        | {successful_attacks}       |
| Failed Attacks            | {failed_attacks}       |
| Skipped Attacks           | {skipped_attacks}       |
+---------------------------+-----------+
""")


[nltk_data] Downloading package averaged_perceptron_tagger_eng to
[nltk_data]     /root/nltk_data...
[nltk_data]   Unzipping taggers/averaged_perceptron_tagger_eng.zip.
textattack: Logging to CSV at path attack_log.csv


Samples originally correct: 477/500
Attack(
  (search_method): GreedyWordSwapWIR(
    (wir_method):  delete
  )
  (goal_function):  UntargetedClassification
  (transformation):  WordSwapEmbedding(
    (max_candidates):  50
    (embedding):  WordEmbedding
  )
  (constraints): 
    (0): WordEmbeddingDistance(
        (embedding):  WordEmbedding
        (min_cos_sim):  0.5
        (cased):  False
        (include_unknown_words):  True
        (compare_against_original):  True
      )
    (1): PartOfSpeech(
        (tagger_type):  nltk
        (tagset):  universal
        (allow_verb_noun_swap):  True
        (compare_against_original):  True
      )
    (2): UniversalSentenceEncoder(
        (metric):  angular
        (threshold):  0.840845057
        (window_size):  15
        (skip_text_shorter_than_window):  True
        (compare_against_original):  True
      )
    (3): RepeatModification
    (4): StopwordModification
  (is_black_box):  True
) 



  0%|          | 0/477 [00:00<?, ?it/s]

1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 55ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 60ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 117ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 126ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 110ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 107ms/step


[Succeeded / Failed / Skipped / Total] 0 / 1 / 0 / 1:   0%|          | 1/477 [00:25<3:24:18, 25.75s/it]

...

1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 63ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 81ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 82ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 79ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 86ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 84ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 88ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 73ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 85ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 73ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 68ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 67ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 87ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 77ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 69ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 73ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 74ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 65ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 72ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 70ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 72ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 67ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 75ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 65ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 70ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 71ms/step
1/1 ━━━━━━━━

[Succeeded / Failed / Skipped / Total] 462 / 14 / 0 / 476: 100%|█████████▉| 476/477 [18:01<00:02,  2.27s/it]

1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 64ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 81ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 84ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 78ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 92ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 82ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 71ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 102ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 111ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 136ms/step


[Succeeded / Failed / Skipped / Total] 463 / 14 / 0 / 477: 100%|██████████| 477/477 [18:04<00:00,  2.27s/it]


+-------------------------------+--------+
| Attack Results                |        |
+-------------------------------+--------+
| Number of successful attacks: | 463    |
| Number of failed attacks:     | 14     |
| Number of skipped attacks:    | 0      |
| Original accuracy:            | 100.0% |
| Accuracy under attack:        | 2.94%  |
| Attack success rate:          | 97.06% |
| Average perturbed word %:     | 5.55%  |
| Average num. words per input: | 108.33 |
| Avg num queries:              | 164.68 |
+-------------------------------+--------+

+---------------------------+-----------+
| Metric                    | Value     |
+---------------------------+-----------+
| Original Accuracy         | 95.40%  |
| Accuracy Under Attack     | 2.80%  |
| Attack Success Rate (ASR) | 97.06%  |
| Successful Attacks        | 463       |
| Failed Attacks            | 14       |
| Skipped Attacks           | 0       |
+---------------------------+-----------+



In [44]:
# Load the model
model = load_model('NLPModel.h5')

In [45]:
# ========================
# STABLE VIRTUAL ADVERSARIAL TRAINING DEFENSE
# ========================
print("\nImplementing Stable Virtual Adversarial Training Defense")

class VATModel(tf.keras.Model):
    def __init__(self, base_model, epsilon=2, xi=1e-6, ip=1):
        super().__init__()
        self.embedding = base_model.layers[0]
        self.encoder = tf.keras.Sequential(base_model.layers[1:])
        self.epsilon = epsilon  # Reduced perturbation
        self.xi = xi            # Smaller step size
        self.ip = ip            # Fewer power iterations

    def _normalize(self, x):
        """Numerically stable normalization"""
        norms = tf.sqrt(tf.reduce_mean(tf.square(x), axis=-1, keepdims=True))
        return x / (norms + 1e-10)

    def call(self, inputs, training=False):
        x = self.embedding(inputs)
        return self.encoder(x, training=training)

    def train_step(self, data):
        x, y = data

        # Original forward pass
        with tf.GradientTape() as tape:
            y_pred = self(x, training=True)
            ce_loss = self.compiled_loss(y, y_pred)
        y_pred_detached = tf.stop_gradient(y_pred)

        # Adversarial perturbation calculation
        embeddings = self.embedding(x)
        d = tf.random.normal(tf.shape(embeddings))

        # Find perturbation direction
        for _ in range(self.ip):
            d = self.xi * self._normalize(d)
            with tf.GradientTape() as tape_adv:
                tape_adv.watch(d)
                perturbed_embeds = embeddings + d
                y_hat = self.encoder(perturbed_embeds, training=True)

                y_pred_detached_clipped = tf.clip_by_value(y_pred_detached, 1e-7, 1-1e-7)
                y_hat_clipped = tf.clip_by_value(y_hat, 1e-7, 1-1e-7)
                kl_loss = tf.reduce_mean(
                    y_hat_clipped * tf.math.log(y_hat_clipped/y_pred_detached_clipped) +
                    (1 - y_hat_clipped) * tf.math.log((1 - y_hat_clipped)/(1 - y_pred_detached_clipped))
              )


            grad = tape_adv.gradient(kl_loss, d)
            d = tf.stop_gradient(self.xi * self._normalize(grad))

        # Apply final perturbation
        r_adv = self.epsilon * self._normalize(d)

        # Adversarial training step
        with tf.GradientTape() as tape_total:
            y_pred_adv = self.encoder(embeddings + r_adv, training=True)

            y_pred_detached_clipped = tf.clip_by_value(y_pred_detached, 1e-7, 1-1e-7)
            y_pred_adv_clipped = tf.clip_by_value(y_pred_adv, 1e-7, 1-1e-7)
            vat_loss = tf.reduce_mean(
                y_pred_detached_clipped * tf.math.log(y_pred_detached_clipped/y_pred_adv_clipped) +
            (1 - y_pred_detached_clipped) * tf.math.log((1 - y_pred_detached_clipped)/(1 - y_pred_adv_clipped))
        )


            # Balanced loss weighting
            total_loss = ce_loss + 1 * vat_loss  # Reduced VAT weight

        # Gradient handling
        gradients = tape_total.gradient(total_loss, self.trainable_variables)
        gradients = [
            tf.clip_by_value(g, -0.5, 0.5) if g is not None
            else tf.zeros_like(v)
            for g, v in zip(gradients, self.trainable_variables)
        ]

        self.optimizer.apply_gradients(zip(gradients, self.trainable_variables))

        # Update metrics
        self.compiled_metrics.update_state(y, y_pred)
        return {m.name: m.result() for m in self.metrics}


def evaluate_model(model, X, y, batch_size=512):
    """Evaluate model on clean data"""
    if len(X) == 0:
        return 0.0
    _, acc = model.evaluate(X, y, batch_size=batch_size, verbose=0)
    return acc

def evaluate_adversarial(defense_model, attack_results, tokenizer, max_length):
    """Evaluate defense on all attack attempts (successful + failed), as in TextAttack."""
    perturbed_texts = []
    true_labels = []
    for res in attack_results:
        # For both successful and failed, use the perturbed text (for skipped, you may skip)
        if hasattr(res, "perturbed_result") and hasattr(res.perturbed_result, "attacked_text"):
            perturbed_texts.append(res.perturbed_result.attacked_text.text)
            true_labels.append(res.original_result.ground_truth_output)
        elif hasattr(res, "original_result") and hasattr(res.original_result, "attacked_text"):
            # For failed attacks, perturbed text may be same as original
            perturbed_texts.append(res.original_result.attacked_text.text)
            true_labels.append(res.original_result.ground_truth_output)
        else:
            continue  # Skip if missing info

    # Convert to model inputs
    sequences = tokenizer.texts_to_sequences(perturbed_texts)
    adv_x = pad_sequences(sequences, maxlen=max_length, padding='post', truncating='post')

    # Get predictions
    defense_preds = defense_model.predict(adv_x, verbose=0)
    defense_adv_acc = np.mean((defense_preds > 0.5).flatten() == true_labels)

    return defense_adv_acc

# Initialize and compile with learning rate schedule
vat_model = VATModel(model, epsilon=2.0, xi=1e-6, ip=1)
vat_model.compile(
    optimizer=tf.keras.optimizers.Adam(
        learning_rate=tf.keras.optimizers.schedules.CosineDecay(
            initial_learning_rate=1e-4,
            decay_steps=1000
        )
    ),
    loss='binary_crossentropy',
    metrics=['accuracy']
)

# Train with validation split
print("Training with Virtual Adversarial Defense...")
history = vat_model.fit(
    tr_x[:5000], tr_y[:5000],
    validation_split=0.2,
    epochs=8,
    batch_size=32,
)
model = load_model('NLPModel.h5')

print("\n=== Adversarial Evaluation ===")
defense_adv_acc = evaluate_adversarial(vat_model, attack_results, tokenizer, max_length)
orig_adv_acc = evaluate_adversarial(model, attack_results, tokenizer, max_length)

# Now print results using the same denominator as TextAttack/paper
successful_attacks = sum(1 for res in attack_results if isinstance(res, SuccessfulAttackResult))
skipped_attacks = sum(1 for res in attack_results if isinstance(res, SkippedAttackResult))
failed_attacks = sum(1 for res in attack_results if isinstance(res, FailedAttackResult))
num_attempted = successful_attacks + failed_attacks  # Only those attempted (not skipped)

asr_original = 1 - orig_adv_acc
asr_defended = 1 - defense_adv_acc
asr_reduction = asr_original - asr_defended

print(f"""
+---------------------------+-----------+
| Metric                    | Value     |
+---------------------------+-----------+
| Original Accuracy         | {orig_adv_acc:.2%}  |
| Defended Accuracy         | {defense_adv_acc:.2%}  |
| Original Model ASR        | {asr_original:.2%}  |
| Defended Model ASR        | {asr_defended:.2%}  |
| ASR Reduction             | {asr_reduction:.2%}  |
| Successful Attacks        | {successful_attacks} |
| Failed Attacks            | {failed_attacks}     |
| Skipped Attacks           | {skipped_attacks}    |
| Total Attempted           | {num_attempted}      |
+---------------------------+-----------+
""")


Implementing Stable Virtual Adversarial Training Defense
Training with Virtual Adversarial Defense...
Epoch 1/8
125/125 ━━━━━━━━━━━━━━━━━━━━ 55s 373ms/step - accuracy: 0.9718 - loss: 0.5133 - val_accuracy: 0.9560 - val_loss: 0.1404
Epoch 2/8
125/125 ━━━━━━━━━━━━━━━━━━━━ 83s 378ms/step - accuracy: 0.9658 - loss: 0.5229 - val_accuracy: 0.9580 - val_loss: 0.1577
Epoch 3/8
125/125 ━━━━━━━━━━━━━━━━━━━━ 82s 383ms/step - accuracy: 0.9670 - loss: 0.5412 - val_accuracy: 0.9550 - val_loss: 0.1570
Epoch 4/8
125/125 ━━━━━━━━━━━━━━━━━━━━ 81s 373ms/step - accuracy: 0.9709 - loss: 0.5202 - val_accuracy: 0.9560 - val_loss: 0.1554
Epoch 5/8
125/125 ━━━━━━━━━━━━━━━━━━━━ 82s 374ms/step - accuracy: 0.9681 - loss: 0.5235 - val_accuracy: 0.9570 - val_loss: 0.1546
Epoch 6/8
125/125 ━━━━━━━━━━━━━━━━━━━━ 82s 373ms/step - accuracy: 0.9608 - loss: 0.5247 - val_accuracy: 0.9570 - val_loss: 0.1556
Epoch 7/8
125/125 ━━━━━━━━━━━━━━━━━━━━ 82s 371ms/step - accuracy: 0.9659 - loss: 0.5296 - val_accuracy: 0.9570 - val_


=== Adversarial Evaluation ===

+---------------------------+-----------+
| Metric                    | Value     |
+---------------------------+-----------+
| Original Accuracy         | 2.94%  |
| Defended Accuracy         | 46.12%  |
| Original Model ASR        | 97.06%  |
| Defended Model ASR        | 53.88%  |
| ASR Reduction             | 43.19%  |
| Successful Attacks        | 463 |
| Failed Attacks            | 14     |
| Skipped Attacks           | 0    |
| Total Attempted           | 477      |
+---------------------------+-----------+

